In [110]:
from datetime import datetime

from risk_manager import RiskManager
from paper_trading_engine import PaperTradingEngine
from trade_logger import TradeLogger

from strategies.dca_strategy import DCAStrategy
from strategies.atr_strategy import ATRStrategy

from telegram_notifier import TelegramNotifier
from llm_advisor import LLMAdvisor

class TradingAgent:
    """
    Main orchestrator for the Bitcoin trading system.

    Responsibilities:
        1. Receive market data
        2. Run trading strategies
        3. Send proposed trades to RiskManager
        4. Execute approved trades using PaperTradingEngine
        5. Record trades using TradeLogger

    IMPORTANT:
        This version is PAPER TRADING ONLY.
        It does not place real Coinbase orders.
    """

    def __init__(self, config):
        """
        Initialize the trading agent using configuration
        loaded from Google Sheets.
        """

        self.config = config

        self.risk = RiskManager(
            total_budget_usd=config["budget_usd"],
            max_trade_usd=config["max_trade_usd"],
            max_position_usd=config["max_position_usd"],
            global_stop_loss_pct=config["global_stop_loss_pct"],
            max_active_trades=config["max_active_trades"],
            paper_trading=config["paper_trading"]
        )

        self.engine = PaperTradingEngine(
            initial_cash_usd=config["budget_usd"],
            trading_fee_pct=config["trading_fee_pct"]
        )

        self.dca = DCAStrategy(
            buy_amount_usd=config["dca_buy_amount_usd"],
            drop_pct=config["dca_drop_pct"],
            enabled=config["dca_enabled"]
        )

        self.atr = ATRStrategy(
            atr_multiplier=config["atr_multiplier"],
            enabled=config["atr_enabled"]
        )

        self.logger = TradeLogger()
        self.paper_trading = config['paper_trading']
        self.telegram = TelegramNotifier()
        self.last_price = None
        self.last_signal = None

        if self.config["llm_enabled"]:
            self.llm = LLMAdvisor()
        else:
            self.llm = None

    # =========================================================
    # MARKET UPDATE
    # =========================================================

    def process_market_data(
        self,
        price,
        atr=None,
        rsi=None,
        macd=None,
        macd_signal=None,
        volume_ratio=None
    ):
        """
        Process a new market observation.

        Parameters
        ----------
        price : float
            Current BTC price.

        atr : float, optional
            ATR indicator.

        rsi : float, optional
            RSI indicator.

        macd : float, optional
            MACD value.

        macd_signal : float, optional
            MACD signal line.

        volume_ratio : float, optional
            Current volume / average volume.

        Returns
        -------
        dict
            Summary of decisions and actions.
        """

        price = float(price)

        self.last_price = price

        # Update paper engine price
        self.engine.update_price(price)

        results = {
            "timestamp": datetime.now().isoformat(),
            "price": price,
            "dca": None,
            "atr": None,
            "trade": None
        }

        # -----------------------------------------------------
        # 1. Check global portfolio risk
        # -----------------------------------------------------

        global_risk = self.risk.check_global_stop(price)

        if not global_risk["approved"]:

            results["risk"] = global_risk

            print(
                "Trading paused:",
                global_risk["reason"]
            )

            return results

        # -----------------------------------------------------
        # 2. Run DCA strategy
        # -----------------------------------------------------

        dca_decision = self.dca.should_buy(price)

        results["dca"] = dca_decision
        atr_decision = None

        if (
            atr is not None
            and rsi is not None
            and macd is not None
            and macd_signal is not None
            and volume_ratio is not None
        ):
        
            atr_decision = self.atr.generate_entry_signal(
                current_price=price,
                atr=atr,
                rsi=rsi,
                macd=macd,
                macd_signal=macd_signal,
                volume_ratio=volume_ratio
            )
        
            results["atr"] = atr_decision
        
        
        # -----------------------------------------------------
        # 4. Run LLM analysis
        # -----------------------------------------------------
        
        llm_decision = None
        
        if (
            self.config.get("llm_enabled", False)
            and self.llm is not None
        ):
        
            llm_decision = self.llm.analyze_market(
                price=price,
                rsi=rsi,
                macd=macd,
                macd_signal=macd_signal,
                atr=atr,
                volume_ratio=volume_ratio
            )
        
            results["llm"] = llm_decision
            
        hybrid_recommendation = "HOLD"
        
        if llm_decision is not None:
        
            llm_available = llm_decision.get(
                "llm_available",
                False
            )
        
            llm_confidence = float(
                llm_decision.get(
                    "confidence",
                    0.0
                )
            )
        
            llm_recommendation = llm_decision.get(
                "recommendation",
                "HOLD"
            )
        
            min_confidence = float(
                self.config.get(
                    "llm_min_confidence",
                    0.70
                )
            )
        
            if (
                llm_available
                and llm_confidence >= min_confidence
            ):
        
                hybrid_recommendation = (
                    llm_recommendation
                )
        
            else:
        
                # LLM unavailable or not confident enough.
                # Fall back to deterministic strategies.
                hybrid_recommendation = "AUTO"
        
        else:
        
            hybrid_recommendation = "AUTO"
        
        
        results["hybrid_recommendation"] = (
            hybrid_recommendation
        )

        if (
            dca_decision.get("action") == "BUY"
            and (
                hybrid_recommendation == "DCA"
                or hybrid_recommendation == "AUTO"
            )
        ):

            trade_amount = dca_decision.get(
                "amount_usd",
                self.dca.buy_amount_usd
            )

            approval = self.risk.approve_trade(
                trade_amount_usd=trade_amount,
                btc_price=price,
                trade_type="DCA"
            )

            if approval["approved"]:

                trade = self.engine.buy_btc(
                    usd_amount=trade_amount,
                    btc_price=price,
                    strategy="DCA",
                    reason=dca_decision.get(
                        "reason",
                        "DCA buy signal"
                    )
                )

                # Tell DCA strategy that a buy occurred
                self.dca.record_buy(price)

                # Log trade
                self.logger.log_trade(trade)

                self.telegram.send_trade_notification(trade)

                results["trade"] = trade

                print(
                    f"DCA BUY executed: "
                    f"${trade_amount:,.2f} "
                    f"at ${price:,.2f}"
                )

            else:

                print(
                    "DCA trade rejected:",
                    approval["reason"]
                )

        return results

    # =========================================================
    # SELL
    # =========================================================

    def execute_sell(
        self,
        btc_quantity,
        strategy="ATR",
        reason="Stop-loss"
    ):
        """
        Execute a simulated BTC sale.

        The sale is performed through the paper trading engine.
        """

        if self.last_price is None:

            raise ValueError(
                "No current BTC price available."
            )

        if btc_quantity <= 0:

            raise ValueError(
                "BTC quantity must be greater than zero."
            )

        trade = self.engine.sell_btc(
            btc_quantity=btc_quantity,
            btc_price=self.last_price,
            strategy=strategy,
            reason=reason
        )

        self.logger.log_trade(trade)
        trade = self.engine.sell_btc(
            btc_quantity=btc_quantity,
            btc_price=self.last_price,
            strategy=strategy,
            reason=reason
        )
        
        self.logger.log_trade(trade)
        
        self.telegram.send_trade_notification(trade)
        
        return trade
        

    # =========================================================
    # PORTFOLIO
    # =========================================================

    def get_portfolio(self):
        """
        Return current portfolio summary.
        """

        return self.engine.get_portfolio_summary(
            self.last_price
        )

    # =========================================================
    # TRADE HISTORY
    # =========================================================

    def get_trade_history(self):
        """
        Return all recorded trades.
        """

        return self.logger.get_trades()

    # =========================================================
    # STATUS
    # =========================================================

    def get_status(self):
        """
        Return current system status.
        """

        return {
            "timestamp": datetime.now().isoformat(),
            "last_price": self.last_price,
            "paper_trading": self.paper_trading,
            "portfolio": self.get_portfolio(),
            "trades": len(
                self.logger.get_trades()
            )
        }

In [119]:
from config_loader import (
    get_google_sheet_config,
    convert_config_values
)

rows = get_google_sheet_config()

config = convert_config_values(rows)

print(config)

{'budget_usd': 10000, 'dca_enabled': True, 'dca_drop_pct': 3, 'dca_amount': 500, 'dca_interval_hours': 24, 'atr_enabled': True, 'atr_period': 14, 'atr_multiplier': 1.5, 'max_position_pct': 30, 'portfolio_stop_pct': 25, 'trading_mode': 'hybrid', 'monitoring_interval_minutes': 30, 'paper_trading': True, 'llm_enabled': True, 'max_position_usd': 2000, 'max_trade_usd': 2000, 'global_stop_loss_pct': 50, 'max_active_trades': 20, 'trading_fee_pct': 0.0015, 'dca_buy_amount_usd': 500, 'llm_min_confidence': 0.6}


In [61]:
agent.process_market_data(
    price=100000
)

DCA trade rejected: Trade amount $5,000.00 exceeds maximum trade size $2,000.00


{'timestamp': '2026-08-31T09:54:05.215454',
 'price': 100000.0,
 'dca': {'action': 'BUY',
  'amount_usd': 5000.0,
  'price': 100000.0,
  'reason': 'Initial DCA purchase'},
 'atr': None,
 'trade': None}

In [62]:
result = agent.process_market_data(
    price=97000
)

print(result)

DCA trade rejected: Trade amount $5,000.00 exceeds maximum trade size $2,000.00
{'timestamp': '2026-08-31T09:54:42.327506', 'price': 97000.0, 'dca': {'action': 'BUY', 'amount_usd': 5000.0, 'price': 97000.0, 'reason': 'Initial DCA purchase'}, 'atr': None, 'trade': None}


In [ ]:
result = agent.process_market_data(
    price=97000
)

print(result)

In [64]:
print(agent.get_status())

{'timestamp': '2026-08-31T09:56:20.641722', 'last_price': 97000.0, 'paper_trading': True, 'portfolio': {'timestamp': '2026-08-31T09:56:20.641823', 'cash_usd': 10000.0, 'btc_quantity': 0.0, 'btc_price': 97000.0, 'btc_value': 0.0, 'portfolio_value': 10000.0, 'initial_value': 10000.0, 'total_pnl': 0.0, 'return_pct': 0.0, 'realized_pnl': 0.0, 'unrealized_pnl': 0.0, 'active_positions': 0, 'total_trades': 0}, 'trades': 3}


In [50]:
agent = TradingAgent(config)
agent.process_market_data(
    price=110000
)

DCA trade rejected: Trade amount $5,000.00 exceeds maximum trade size $2,000.00


{'timestamp': '2026-08-31T09:37:19.588686',
 'price': 110000.0,
 'dca': {'action': 'BUY',
  'amount_usd': 5000.0,
  'price': 110000.0,
  'reason': 'Initial DCA purchase'},
 'atr': None,
 'trade': None}

In [53]:
agent.process_market_data(
    price=100000
)

DCA trade rejected: Trade amount $5,000.00 exceeds maximum trade size $2,000.00


{'timestamp': '2026-08-31T09:38:46.274058',
 'price': 100000.0,
 'dca': {'action': 'BUY',
  'amount_usd': 5000.0,
  'price': 100000.0,
  'reason': 'Initial DCA purchase'},
 'atr': None,
 'trade': None}

In [54]:
print(agent.get_status())

{'timestamp': '2026-08-31T09:38:53.072671', 'last_price': 100000.0, 'paper_trading': True, 'portfolio': {'timestamp': '2026-08-31T09:38:53.072713', 'cash_usd': 10000.0, 'btc_quantity': 0.0, 'btc_price': 100000.0, 'btc_value': 0.0, 'portfolio_value': 10000.0, 'initial_value': 10000.0, 'total_pnl': 0.0, 'return_pct': 0.0, 'realized_pnl': 0.0, 'unrealized_pnl': 0.0, 'active_positions': 0, 'total_trades': 0}, 'trades': 3}


In [55]:
history = agent.get_trade_history()

for trade in history:
    print(trade)

{'timestamp': '2026-08-31T09:00:00', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '97000', 'btc_quantity': '0.00515464', 'usd_amount': '500', 'fee': '3', 'realized_pnl': '', 'reason': 'BTC dropped 3%'}
{'timestamp': '2026-08-31T08:52:51.910907', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '100000.0', 'btc_quantity': '0.005', 'usd_amount': '500.0', 'fee': '3.0', 'realized_pnl': '', 'reason': 'Initial DCA purchase'}
{'timestamp': '2026-08-31T08:53:13.398356', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '97000.0', 'btc_quantity': '0.005154639175257732', 'usd_amount': '500.0', 'fee': '3.0', 'realized_pnl': '', 'reason': 'BTC dropped 3.00% since last DCA purchase'}


In [63]:
import pandas as pd

df = pd.read_csv(
    "/Users/melaniequ/bitcoin-trader/logs/trades.csv"
)

df

,timestamp,action,strategy,btc_price,btc_quantity,usd_amount,fee,realized_pnl,reason
0,2026-08-31T09:00:00,BUY,DCA,97000.0,0.005155,500.0,3.0,NaN,BTC dropped 3%
1,2026-08-31T08:52:51.910907,BUY,DCA,100000.0,0.005000,500.0,3.0,NaN,Initial DCA purchase
2,2026-08-31T08:53:13.398356,BUY,DCA,97000.0,0.005155,500.0,3.0,NaN,BTC dropped 3.00% since last DCA purchase


In [4]:
logger = TradeLogger()

print(logger.get_log_path())

/Users/melaniequ/bitcoin-trader/logs/trades.csv


In [5]:
test_trade = {
    "timestamp": "2026-08-31T09:00:00",
    "action": "BUY",
    "strategy": "DCA",
    "btc_price": 97000,
    "btc_quantity": 0.00515464,
    "usd_amount": 500,
    "fee": 3,
    "realized_pnl": "",
    "reason": "BTC dropped 3%"
}

logger.log_trade(test_trade)

print("Trade logged!")

Trade logged!


In [6]:
print(logger.get_trades())

[{'timestamp': '2026-08-31T09:00:00', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '97000', 'btc_quantity': '0.00515464', 'usd_amount': '500', 'fee': '3', 'realized_pnl': '', 'reason': 'BTC dropped 3%'}]


In [7]:
import pandas as pd

df = pd.read_csv(
    "/Users/melaniequ/bitcoin-trader/logs/trades.csv"
)

df

,timestamp,action,strategy,btc_price,btc_quantity,usd_amount,fee,realized_pnl,reason
0,2026-08-31T09:00:00,BUY,DCA,97000,0.005155,500,3,NaN,BTC dropped 3%


In [8]:
agent = TradingAgent(
    budget_usd=10000,
    max_trade_usd=1000,
    max_position_usd=5000,
    global_stop_loss_pct=25,
    max_active_trades=3,
    dca_buy_amount_usd=500,
    dca_drop_pct=3.0,
    atr_multiplier=1.5,
    trading_fee_pct=0.006,
    paper_trading=True
)

In [9]:
result = agent.process_market_data(
    price=100000
)

print(result)

DCA BUY executed: $500.00 at $100,000.00
{'timestamp': '2026-08-31T08:52:51.910776', 'price': 100000.0, 'dca': {'action': 'BUY', 'amount_usd': 500.0, 'price': 100000.0, 'reason': 'Initial DCA purchase'}, 'atr': None, 'trade': {'timestamp': '2026-08-31T08:52:51.910907', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 100000.0, 'btc_quantity': 0.005, 'usd_amount': 500.0, 'fee': 3.0, 'reason': 'Initial DCA purchase'}}


In [10]:
result = agent.process_market_data(
    price=97000
)

print(result)

DCA BUY executed: $500.00 at $97,000.00
{'timestamp': '2026-08-31T08:53:13.398290', 'price': 97000.0, 'dca': {'action': 'BUY', 'amount_usd': 500.0, 'price': 97000.0, 'price_change_pct': -3.0, 'reason': 'BTC dropped 3.00% since last DCA purchase'}, 'atr': None, 'trade': {'timestamp': '2026-08-31T08:53:13.398356', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 97000.0, 'btc_quantity': 0.005154639175257732, 'usd_amount': 500.0, 'fee': 3.0, 'reason': 'BTC dropped 3.00% since last DCA purchase'}}


In [11]:
portfolio = agent.get_portfolio()

print(portfolio)

{'timestamp': '2026-08-31T08:53:54.505712', 'cash_usd': 8994.0, 'btc_quantity': 0.010154639175257732, 'btc_price': 97000.0, 'btc_value': 985.0, 'portfolio_value': 9979.0, 'initial_value': 10000.0, 'total_pnl': -21.0, 'return_pct': -0.21, 'realized_pnl': 0.0, 'unrealized_pnl': -15.0, 'active_positions': 2, 'total_trades': 2}


In [12]:
history = agent.get_trade_history()

for trade in history:
    print(trade)

{'timestamp': '2026-08-31T09:00:00', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '97000', 'btc_quantity': '0.00515464', 'usd_amount': '500', 'fee': '3', 'realized_pnl': '', 'reason': 'BTC dropped 3%'}
{'timestamp': '2026-08-31T08:52:51.910907', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '100000.0', 'btc_quantity': '0.005', 'usd_amount': '500.0', 'fee': '3.0', 'realized_pnl': '', 'reason': 'Initial DCA purchase'}
{'timestamp': '2026-08-31T08:53:13.398356', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': '97000.0', 'btc_quantity': '0.005154639175257732', 'usd_amount': '500.0', 'fee': '3.0', 'realized_pnl': '', 'reason': 'BTC dropped 3.00% since last DCA purchase'}


In [13]:
import pandas as pd

df = pd.read_csv(
    "/Users/melaniequ/bitcoin-trader/logs/trades.csv"
)

df

,timestamp,action,strategy,btc_price,btc_quantity,usd_amount,fee,realized_pnl,reason
0,2026-08-31T09:00:00,BUY,DCA,97000.0,0.005155,500.0,3.0,NaN,BTC dropped 3%
1,2026-08-31T08:52:51.910907,BUY,DCA,100000.0,0.005000,500.0,3.0,NaN,Initial DCA purchase
2,2026-08-31T08:53:13.398356,BUY,DCA,97000.0,0.005155,500.0,3.0,NaN,BTC dropped 3.00% since last DCA purchase


In [65]:
from telegram_notifier import TelegramNotifier

telegram = TelegramNotifier()

result = telegram.send_message(
    "🤖 TEST — Bitcoin Trading Agent Telegram connection is working."
)

print(result)

{'ok': True, 'result': {'message_id': 7, 'from': {'id': 8946671149, 'is_bot': True, 'first_name': 'melanie_bitcoin_trading_bot', 'username': 'melanie_bitcoin_trading_bot'}, 'chat': {'id': 8817817440, 'first_name': 'Melanie', 'last_name': 'Qu', 'type': 'private'}, 'date': 1788184868, 'text': '🤖 TEST — Bitcoin Trading Agent Telegram connection is working.'}}


In [66]:
test_trade = {
    "action": "BUY",
    "strategy": "DCA",
    "btc_price": 97000,
    "btc_quantity": 0.00515464,
    "usd_amount": 500,
    "fee": 3,
    "reason": "BTC dropped 3%"
}

telegram.send_trade_notification(test_trade)

{'ok': True,
 'result': {'message_id': 8,
  'from': {'id': 8946671149,
   'is_bot': True,
   'first_name': 'melanie_bitcoin_trading_bot',
   'username': 'melanie_bitcoin_trading_bot'},
  'chat': {'id': 8817817440,
   'first_name': 'Melanie',
   'last_name': 'Qu',
   'type': 'private'},
  'date': 1788184914,
  'text': '🟢 BTC BUY\n\nStrategy: DCA\nPrice: $97,000.00\nBTC: 0.00515464\nAmount: $500.00\nFee: $3.00\nReason: BTC dropped 3%'}}

In [70]:
from trading_agent import TradingAgent

agent = TradingAgent(config)
print(agent.telegram)

In [71]:
result = agent.process_market_data(price=100000)

print(result)

DCA trade rejected: Trade amount $5,000.00 exceeds maximum trade size $2,000.00
{'timestamp': '2026-08-31T10:04:06.637469', 'price': 100000.0, 'dca': {'action': 'BUY', 'amount_usd': 5000.0, 'price': 100000.0, 'reason': 'Initial DCA purchase'}, 'atr': None, 'trade': None}


In [74]:
result = agent.process_market_data(price=7000)

print(result)

DCA trade rejected: Trade amount $5,000.00 exceeds maximum trade size $2,000.00
{'timestamp': '2026-08-31T10:06:41.986066', 'price': 7000.0, 'dca': {'action': 'BUY', 'amount_usd': 5000.0, 'price': 7000.0, 'reason': 'Initial DCA purchase'}, 'atr': None, 'trade': None}


In [73]:
print(result["trade"])

None


In [104]:
import inspect
agent = TradingAgent(config)
print(inspect.getsource(agent.process_market_data))

    def process_market_data(
        self,
        price,
        atr=None,
        rsi=None,
        macd=None,
        macd_signal=None,
        volume_ratio=None
    ):
        """
        Process a new market observation.

        Parameters
        ----------
        price : float
            Current BTC price.

        atr : float, optional
            ATR indicator.

        rsi : float, optional
            RSI indicator.

        macd : float, optional
            MACD value.

        macd_signal : float, optional
            MACD signal line.

        volume_ratio : float, optional
            Current volume / average volume.

        Returns
        -------
        dict
            Summary of decisions and actions.
        """

        price = float(price)

        self.last_price = price

        # Update paper engine price
        self.engine.update_price(price)

        results = {
            "timestamp": datetime.now().isoformat(),
            "price": price,
          

In [111]:
agent = TradingAgent(config)
result = agent.process_market_data(
    price=100000,
    atr=1500,
    rsi=55,
    macd=100,
    macd_signal=80,
    volume_ratio=1.2
)

print(result)

DCA BUY executed: $500.00 at $100,000.00
{'timestamp': '2026-08-31T12:36:16.031859', 'price': 100000.0, 'dca': {'action': 'BUY', 'amount_usd': 500.0, 'price': 100000.0, 'reason': 'Initial DCA purchase'}, 'atr': {'action': 'BUY', 'price': 100000.0, 'atr': 1500.0, 'stop_price': 97750.0, 'atr_multiplier': 1.5, 'confirmations': 3, 'reason': 'RSI supports momentum; MACD bullish; volume breakout'}, 'trade': {'timestamp': '2026-08-31T12:36:23.891536', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 100000.0, 'btc_quantity': 0.005, 'usd_amount': 500.0, 'fee': 7500.0, 'reason': 'Initial DCA purchase'}, 'llm': {'market_regime': 'bullish', 'recommendation': 'SWING', 'confidence': 0.7, 'reason': 'Momentum indicators are positive: MACD (100) is above its signal (80) and RSI at 55 is neutral-to-bullish. Volume is slightly elevated (volume_ratio 1.2), supporting the move. ATR relative to price (~1.5%) indicates moderate volatility, not extreme. Some moving-average and recent-return data are missing,

In [120]:
agent = TradingAgent(config)
result = agent.process_market_data(
    price=100000,
    rsi=55,
    macd=100,
    macd_signal=80,
    atr=1500,
    volume_ratio=1.2
)

print(result)

DCA BUY executed: $500.00 at $100,000.00
{'timestamp': '2026-08-31T12:45:51.371617', 'price': 100000.0, 'dca': {'action': 'BUY', 'amount_usd': 500.0, 'price': 100000.0, 'reason': 'Initial DCA purchase'}, 'atr': {'action': 'BUY', 'price': 100000.0, 'atr': 1500.0, 'stop_price': 97750.0, 'atr_multiplier': 1.5, 'confirmations': 3, 'reason': 'RSI supports momentum; MACD bullish; volume breakout'}, 'trade': {'timestamp': '2026-08-31T12:46:03.730778', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 100000.0, 'btc_quantity': 0.005, 'usd_amount': 500.0, 'fee': 0.75, 'reason': 'Initial DCA purchase'}, 'llm': {'market_regime': 'bullish', 'recommendation': 'SWING', 'confidence': 0.65, 'reason': 'MACD is above its signal line and RSI is neutral-to-bullish (55), with volume slightly elevated (volume_ratio 1.2) suggesting bullish momentum with acceptable volatility (ATR ≈ 1.5% of price). Data points are limited (several indicators null), so conviction is moderate.', 'suggested_atr_multiplier': 1.5},

In [121]:
print("Hybrid:", result.get("hybrid_recommendation"))
print("LLM:", result.get("llm"))
print("Trade:", result.get("trade"))

Hybrid: AUTO
LLM: {'market_regime': 'bullish', 'recommendation': 'SWING', 'confidence': 0.65, 'reason': 'MACD is above its signal line and RSI is neutral-to-bullish (55), with volume slightly elevated (volume_ratio 1.2) suggesting bullish momentum with acceptable volatility (ATR ≈ 1.5% of price). Data points are limited (several indicators null), so conviction is moderate.', 'suggested_atr_multiplier': 1.5}
Trade: {'timestamp': '2026-08-31T12:46:03.730778', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 100000.0, 'btc_quantity': 0.005, 'usd_amount': 500.0, 'fee': 0.75, 'reason': 'Initial DCA purchase'}


In [92]:
agent = TradingAgent(config)

result = agent.process_market_data(
    price=110000
)

print(result)
print(agent.get_status())

result = agent.process_market_data(
    price=100000
)

print(result["trade"])

DCA BUY executed: $500.00 at $110,000.00
{'timestamp': '2026-08-31T10:26:36.895764', 'price': 110000.0, 'dca': {'action': 'BUY', 'amount_usd': 500.0, 'price': 110000.0, 'reason': 'Initial DCA purchase'}, 'atr': None, 'trade': {'timestamp': '2026-08-31T10:26:36.895971', 'action': 'BUY', 'strategy': 'DCA', 'btc_price': 110000.0, 'btc_quantity': 0.004545454545454545, 'usd_amount': 500.0, 'fee': 7500.0, 'reason': 'Initial DCA purchase'}}
{'timestamp': '2026-08-31T10:26:37.369866', 'last_price': 110000.0, 'paper_trading': True, 'portfolio': {'timestamp': '2026-08-31T10:26:37.369919', 'cash_usd': 2000.0, 'btc_quantity': 0.004545454545454545, 'btc_price': 110000.0, 'btc_value': 500.0, 'portfolio_value': 2500.0, 'initial_value': 10000.0, 'total_pnl': -7500.0, 'return_pct': -75.0, 'realized_pnl': 0.0, 'unrealized_pnl': 0.0, 'active_positions': 1, 'total_trades': 1}, 'trades': 4}


ValueError: Insufficient cash after trading fee.

In [93]:
agent = TradingAgent(config)

agent.process_market_data(price=100000)

result = agent.process_market_data(price=97000)

print(result)
print(result["trade"])

DCA BUY executed: $500.00 at $100,000.00


ValueError: Insufficient cash after trading fee.